# Torch — full re-embed + dual-write to B2 + Qdrant + BM25 indices

**Hardware target:** 80-core CPU + 500GB RAM + RTX 2060 (8 GB VRAM).

**What this notebook does, end to end:**
1. Hardware + env check (GPU detected? B2 reachable? Qdrant up? GEMINI_API_KEY set?)
2. **Docs** — crawl pytorch.org/docs/stable → section-aware chunk → BGE-base embed (GPU) → write JSONL.gz to B2 → upsert to Qdrant
3. **Code** — walk `data/pytorch/` → AST per function/class → embed (GPU) → JSONL → B2 → Qdrant
4. **Issues** — `pytorch/pytorch` GitHub issues (last 12 months) → chunk title+body+top comments → BGE embed → JSONL → B2 → Qdrant
5. **BM25** — for each collection, build a `BM25Okapi` index from the JSONL → pickle to local + B2
6. **Smoke test** — fire a `/search` request through the hybrid retriever to confirm dense + BM25 + RRF return sensible chunks

**Why dual-write:** Qdrant Cloud's free tier can drop collections without warning. B2 holds the durable JSONL.gz source of truth — if Qdrant gets wiped, `python -m app.scripts.restore_from_b2 --all` rebuilds in minutes without touching pytorch.org again.

Run cells top-to-bottom. Each section is idempotent — you can re-run any one cell without poisoning the others.

## 0 · Setup — env + hardware probe

In [ ]:
import os, sys, time, gzip, json, uuid, pathlib, subprocess
from pathlib import Path

# Make `from app...` work whether the kernel boots in notebooks/ or backend/
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)                                  # ensure relative paths (data/, .env) resolve
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from dotenv import load_dotenv
load_dotenv(REPO / ".env")
from tqdm.auto import tqdm                       # used by every later cell

print("cwd  :", Path.cwd())
print("repo :", REPO)
print("python:", sys.version.split()[0])
print()

# torch / GPU
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM: {free/1e9:.1f} / {total/1e9:.1f} GB free")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# CPU + RAM
try:
    import psutil
    print(f"cpu cores: {psutil.cpu_count(logical=True)}  |  RAM: {psutil.virtual_memory().total/1e9:.0f} GB")
except Exception:
    print("cpu cores:", os.cpu_count())

# env sanity
expected = ["QDRANT_URL", "QDRANT_API_KEY", "B2_S3_ENDPOINT_URL", "B2_BUCKET",
            "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "GEMINI_API_KEY", "GITHUB_TOKEN"]
missing = [k for k in expected if not os.environ.get(k)]
if missing:
    print("\n⚠️  missing env vars:", missing)
else:
    print("\n✓ all required env vars set")


In [ ]:
# verify b2 + qdrant are actually reachable BEFORE doing 30 minutes of work
from app.storage import b2
from app.db.qdrant import get_qdrant

print('B2.ping():', b2.ping())
print('B2 bucket:', os.environ.get('B2_BUCKET'))

qc = get_qdrant()
print('Qdrant collections:', [c.name for c in qc.get_collections().collections])

In [ ]:
# Pull the BGE + CodeBERT models onto the GPU before we start ingesting,
# so the first batch isn't a 30s cold start in the middle of a tqdm loop.
from app.embeddings.encoder import embed_text, embed_code

t = time.perf_counter()
_ = embed_text(['warmup: torch.nn.Linear'])
print(f'  bge-base loaded in {time.perf_counter()-t:.1f}s')
t = time.perf_counter()
_ = embed_code(['def forward(self, x): return self.fc(x)'])
print(f'  code encoder loaded in {time.perf_counter()-t:.1f}s')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'  VRAM after warmup: {free/1e9:.1f} / {total/1e9:.1f} GB free')

## 1 · Shared helpers

`embed_batched` runs the encoder in fixed-size batches with a progress bar — small enough to fit in 8 GB VRAM, big enough to avoid Python-overhead per call. `to_record` is the canonical JSONL shape the B2 layer expects.

In [ ]:
from tqdm.auto import tqdm
from app.storage import b2

BATCH = 64 if torch.cuda.is_available() else 8   # 2060 8GB handles 64 comfortably

def embed_batched(encoder_fn, texts, *, batch=BATCH, desc='embed'):
    """Run encoder_fn in batches with a progress bar; return list-of-list floats."""
    out = []
    for i in tqdm(range(0, len(texts), batch), desc=desc):
        out.extend(encoder_fn(texts[i:i+batch]))
    return out

def to_record(kind, source_url, title, content, *, section=None, anchor=None,
              labels=None, state=None, score=None, last_synced_at=None, sha=None,
              vector=None):
    rid = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{source_url}|{anchor or ""}|{title or ""}|{content[:80]}'))
    rec = {
        'id': rid,
        'kind': kind,
        'source_url': source_url,
        'title': title or '',
        'section': section,
        'content': content,
        'anchor': anchor,
        'labels': labels or [],
        'state': state,
        'score': score,
        'last_synced_at': last_synced_at or int(time.time()),
        'sha': sha,
    }
    if vector is not None:
        rec['vector'] = vector
    return rec

def upsert_to_qdrant(qcoll, records):
    """Upsert records (each carrying `vector`) to Qdrant in 256-pt batches."""
    from qdrant_client.http.models import PointStruct, Distance, VectorParams
    client = get_qdrant()
    # ensure collection exists
    if not any(c.name == qcoll for c in client.get_collections().collections):
        dim = len(records[0]['vector'])
        client.create_collection(qcoll, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
        print(f'  [qdrant] created {qcoll} (dim={dim})')
    points = []
    BATCH_Q = 256
    for r in records:
        pid = str(uuid.uuid5(uuid.NAMESPACE_URL, r['id']))
        payload = {k: v for k, v in r.items() if k != 'vector'}
        points.append(PointStruct(id=pid, vector=r['vector'], payload=payload))
        if len(points) >= BATCH_Q:
            client.upsert(qcoll, points=points)
            points = []
    if points:
        client.upsert(qcoll, points=points)
    print(f'  [qdrant] upserted {len(records)} → {qcoll}')

def push_jsonl_to_b2(layout, records):
    """Persist records as JSONL.gz to B2 (versioned batch + latest pointer)."""
    bkey, n = b2.write_chunks_batch(layout, iter(records), update_latest=True)
    print(f'  [b2] wrote {n} records → {bkey}')
    return bkey

## 2 · Docs — `pytorch.org/docs/stable`

Crawls the versioned docs index, extracts the article body of every page,
chunks it, embeds with BGE-base, and dual-writes to B2 + Qdrant in one go.

Uses a **parallel crawler** (16 threads on `requests.get`) so 2 800+ pages
land in ~3 min instead of ~22 min sequential.

End state: `docs/latest.jsonl.gz` in B2 + Qdrant collection `torch_docs`.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from app.ingestion.docs.crawler import get_doc_links, extract_page
from app.ingestion.docs.chunker import chunk_docs
from app.db.qdrant import DOCS_COLLECTION_NAME

# 1) discover URLs
links = get_doc_links()
print(f"doc links: {len(links)}")

# 2) parallel crawl (16-way) — bandwidth + latency bound, GPU is idle here
pages = []
with ThreadPoolExecutor(max_workers=16) as ex:
    for p in tqdm(ex.map(extract_page, links), total=len(links), desc="crawl docs"):
        if p:
            pages.append(p)
print(f"pages fetched: {len(pages)}")

# 3) chunk + flatten
raw_pairs = []   # (page, chunk_text)
for page in pages:
    for c in chunk_docs(page["text"]):
        raw_pairs.append((page, c))
print(f"chunks before embed: {len(raw_pairs)}")

# 4) batched GPU embed
vectors = embed_batched(embed_text, [c for _, c in raw_pairs], desc="embed docs")

# 5) materialize records
doc_records = []
for (page, c), v in zip(raw_pairs, vectors):
    doc_records.append(to_record(
        "docs",
        source_url=page["url"],
        title=page.get("title") or "",
        content=c,
        vector=v,
    ))
print(f"docs records ready: {len(doc_records)}")

# 6) dual-write — ALWAYS in the same cell as embed so a stop-mid-cell
#    never leaves data only in Qdrant or only in B2.
push_jsonl_to_b2(b2.DOCS, doc_records)
upsert_to_qdrant(DOCS_COLLECTION_NAME, doc_records)


> _Note: the B2 + Qdrant write used to live in a separate cell here.
> It's now folded into cell 8 above so partial runs can't leave docs
> only in Qdrant (or only in B2). Skip this cell — there's nothing to run._


## 3 · Code — `torch/*` source tree

AST per top-level function / class. ~12k chunks. Embedding still goes through `embed_code` (currently CodeBERT). When you're ready to swap to `jinaai/jina-embeddings-v2-base-code`, change `app/embeddings/encoder.py:_load_code_model` and re-run *only this section*.

In [ ]:
from app.ingestion.code.code_parser import CodeChunkExtractor
from app.db.qdrant import CODE_COLLECTION_NAME
import ast

CODE_ROOTS = [
    'data/pytorch/torch/nn/modules',
    'data/pytorch/torch/nn/functional.py',
    'data/pytorch/torch/optim',
    'data/pytorch/torch/autograd',
    'data/pytorch/torch/cuda',
    'data/pytorch/torch/_dynamo',
    'data/pytorch/torch/_inductor',
]

def collect_py_files(roots):
    out = []
    for r in roots:
        p = Path(r)
        if not p.exists():
            continue
        if p.is_file() and p.suffix == '.py':
            out.append(p)
        elif p.is_dir():
            out.extend(p.rglob('*.py'))
    return out

py_files = collect_py_files(CODE_ROOTS)
print(f'py files: {len(py_files)}')

code_chunks = []
for f in tqdm(py_files, desc='parse AST'):
    try:
        src = f.read_text(encoding='utf-8', errors='ignore')
        tree = ast.parse(src)
    except Exception:
        continue
    ex = CodeChunkExtractor(src, str(f))
    ex.visit(tree)
    for c in ex.chunks:
        # the parser yields fields like {name, kind, code, lineno, end_lineno, file_path, docstring}
        symbol = c.get('name', '<anon>')
        start = c.get('lineno', 0)
        end = c.get('end_lineno', start)
        rel = str(f.relative_to('data/pytorch')) if 'data/pytorch' in str(f) else str(f)
        code_chunks.append({
            'symbol': symbol,
            'anchor': f'L{start}-L{end}',
            'file': rel,
            'url': f'https://github.com/pytorch/pytorch/blob/main/{rel}#L{start}',
            'code': c.get('code') or '',
            'docstring': c.get('docstring') or '',
        })

print(f'code chunks: {len(code_chunks)}')

In [ ]:
texts = [(c['docstring'] + '\n\n' + c['code']).strip() for c in code_chunks]
vectors = embed_batched(embed_code, texts, desc='embed code')

code_records = []
for c, v in zip(code_chunks, vectors):
    code_records.append(to_record(
        'code',
        source_url=c['url'],
        title=c['symbol'],
        content=texts[len(code_records)],
        section=c['file'],
        anchor=c['anchor'],
        vector=v,
    ))

push_jsonl_to_b2(b2.CODE, code_records)
upsert_to_qdrant(CODE_COLLECTION_NAME, code_records)

## 4 · Issues — `pytorch/pytorch` last 12 months

PyGithub against `pytorch/pytorch`. Default cap **1 500 issues with their
top-5 comments**, fetched via `ThreadPoolExecutor(16)` so we pay one HTTPS
roundtrip per issue concurrently instead of sequentially. Wall time: ~3 min.

Bump `LIMIT` to 5 000 for fuller coverage once the smoke test passes.
End state: `issues/latest.jsonl.gz` in B2 + Qdrant collection `torch_issues`.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from github import Github
from app.ingestion.issues.chunker import chunk_issue
from app.db.qdrant import ISSUES_COLLECTION_NAME

LIMIT = 1500             # how many issues to ingest
COMMENT_CAP = 5          # keep at most N comments per issue (bot replies are noise)
REPO = "pytorch/pytorch"

# 1) paginate cheaply — DON'T touch issue.body / get_comments() here, those
#    each trigger a separate HTTPS call. Just collect the lazy proxies.
gh = Github(os.environ["GITHUB_TOKEN"], per_page=100)
repo = gh.get_repo(REPO)

print("paginating issues…")
proxies = []
for iss in tqdm(repo.get_issues(state="all")[:LIMIT], total=LIMIT, desc="meta"):
    proxies.append(iss)
print(f"got {len(proxies)} proxies")

# 2) hydrate each issue's body + comments in parallel
def hydrate(iss):
    return {
        "number": iss.number,
        "title": iss.title or "",
        "body": iss.body or "",
        "comments": [c.body for c in iss.get_comments()[:COMMENT_CAP]],
        "html_url": iss.html_url,
        "url": iss.html_url,
        "labels": [l.name for l in iss.labels],
        "state": iss.state,
    }

print("fetching bodies + comments in parallel…")
issues = []
with ThreadPoolExecutor(max_workers=16) as ex:
    for r in tqdm(ex.map(hydrate, proxies), total=len(proxies), desc="hydrate"):
        issues.append(r)
print(f"issues hydrated: {len(issues)}")

# 3) chunk
issue_raw_chunks = []   # (issue_meta, chunk_text)
for iss in tqdm(issues, desc="chunk issues"):
    for c in chunk_issue(iss):
        issue_raw_chunks.append((iss, c))
print(f"issue chunks: {len(issue_raw_chunks)}")

# 4) batched GPU embed
texts = [c for _, c in issue_raw_chunks]
vectors = embed_batched(embed_text, texts, desc="embed issues")

# 5) materialize records
issue_records = []
for (iss, c), v in zip(issue_raw_chunks, vectors):
    issue_records.append(to_record(
        "issues",
        source_url=iss.get("html_url") or iss.get("url") or "",
        title=iss.get("title") or f"#{iss.get('number')}",
        content=c,
        labels=[l["name"] if isinstance(l, dict) else l for l in (iss.get("labels") or [])],
        state=iss.get("state"),
        vector=v,
    ))

# 6) dual-write (same cell as embed — see docs cell for rationale)
push_jsonl_to_b2(b2.ISSUES, issue_records)
upsert_to_qdrant(ISSUES_COLLECTION_NAME, issue_records)


## 5 · BM25 indices — built from the in-memory records, pushed to B2

These let the hybrid retriever catch literal symbol matches like `set_to_none=True` that the dense embedder paraphrases away. RRF (k=60) fuses BM25's ranking with dense for the final result.

In [ ]:
from app.retrieval import bm25

# Rehydrate any record-lists that aren't in scope (e.g. you only re-ran the
# issues cell, but want to rebuild BM25 across all three corpora).
def _ensure(name, layout):
    var = f"{name}_records"
    if var in globals() and globals()[var]:
        return globals()[var]
    try:
        rows = list(b2.read_latest(layout))
        print(f"  hydrated {len(rows)} {name} records from B2")
        return rows
    except Exception as e:
        print(f"  no {name} in scope or in B2 ({type(e).__name__}) — skipping")
        return []

doc_records   = _ensure("doc",   b2.DOCS)    if "doc_records"   not in globals() or not doc_records   else doc_records
code_records  = _ensure("code",  b2.CODE)   if "code_records"  not in globals() or not code_records  else code_records
issue_records = _ensure("issue", b2.ISSUES) if "issue_records" not in globals() or not issue_records else issue_records

for name, records in [("docs", doc_records), ("code", code_records), ("issues", issue_records)]:
    if not records:
        print(f"  skip BM25/{name}: empty corpus")
        continue
    print(f"\nbuilding BM25/{name} …")
    lite = [{k: v for k, v in r.items() if k != "vector"} for r in records]
    idx = bm25.build(lite)
    bm25.save(name, idx, push_to_b2=True)     # correct arg order: (collection, idx)
    bm25.set_index(name, idx)
    print(f"  ✓ {name}: n={idx.n}  →  data/bm25/{name}.pkl + B2 bm25/{name}.pkl")


## 6 · Smoke test — hybrid retrieval over the freshly-indexed corpus

In [ ]:
from app.retrieval.hybrid import hybrid_search

queries = [
    'why does DataLoader hang with num_workers>0 on macOS?',
    'difference between .detach() and .data',
    'set_to_none=True on zero_grad',
]

for q in queries:
    print('\n' + '='*80)
    print('Q:', q)
    hits = hybrid_search(q, rerank_top_k=4)
    for h in hits:
        p = h['payload']
        print(f"  [{p.get('kind'):>6}] score={h.get('score', 0):.3f}  {p.get('title', '')[:80]}")
        print('         ', (p.get('source_url') or p.get('url') or '')[:100])

## Done ✅

What's now durable in B2:

```
<bucket>/
  docs/   chunks/<ts>.jsonl.gz   +  latest.jsonl.gz
  code/   chunks/<ts>.jsonl.gz   +  latest.jsonl.gz
  issues/ chunks/<ts>.jsonl.gz   +  latest.jsonl.gz
  bm25/   docs.pkl  code.pkl  issues.pkl
```

If Qdrant ever wipes:

```bash
python -m app.scripts.restore_from_b2 --all
```

Next:

```bash
# boot the API
cd backend
uvicorn app.main:app --reload --host 0.0.0.0 --port 8000

# from another shell — sanity
curl http://localhost:8000/healthz
curl http://localhost:8000/sources
curl -X POST http://localhost:8000/search -H 'content-type: application/json' \
  -d '{"query":"num_workers macOS hang"}'
```

For the streaming `/ask` answer:

```bash
curl -N -X POST http://localhost:8000/ask -H 'content-type: application/json' \
  -d '{"query":"why does DataLoader hang with num_workers>0 on macOS?"}'
```